# Part 2 revision analyses: inorganic surrogate (re-validated) and organic–inorganic combination with R_emb

**Part I, inorganic surrogate.** The original training script used full-batch updates (one gradient step per
epoch), unstandardized outputs, and the *interpolation test set as the early-stopping set*. Here the surrogate
is retrained with a corrected protocol:
- mini-batches;
- per-column output standardization with masking;
- a separate validation split for early stopping;
- 5 seeds.

The original protocol is also reproduced over 5 seeds for comparison. Further sections:
- per-species, pooled and median errors;
- a comparison with **real Fortran reference output** (cases c005 and c006);
- the bisulfate closed-form pipeline;
- an end-to-end cost benchmark;
- the three Sobol cases, run through the surrogate *and* through AIOMFAC directly.

**Part II, combination.** Option A/C and the OOD guard are rebuilt on the Part 1 release surrogate (R_emb
ensemble, `../part1_organic_surrogate/checkpoints`). The residual model sees the organic through its 16-dimensional R_emb
embedding. The guard threshold is calibrated on a *separate* set of calibration organics and evaluated on
different test organics. Malonic acid and ethanol, the two worked-example organics, are excluded from the
residual training pool.

Inputs (from `inorganic_surrogate_bundle.zip` / `combo_bundle.zip`): `training_data.parquet`, `organics.json`,
`inorganic_component_basis.py`, `generate_training_data.py`, `closed_form_pipeline.py`,
`train_mlp_benchmark.py`, `aiomfac-python/tests/` (Fortran reference dumps), and `CKPT_DIR`.

In [1]:
import sys, os, json, math, time, importlib.util, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
warnings.filterwarnings("ignore")
HERE = Path.cwd(); sys.path.insert(0, str(HERE)); sys.path.insert(0, str(HERE / "aiomfac-python" / "tests"))
CKPT_DIR = Path(os.environ.get("CKPT_DIR", HERE.parent / "part1_organic_surrogate" / "checkpoints"))
import aiomfac_py.s2as._mapping as _m; _m.print = lambda *a, **k: None
from aiomfac_py import ActivityModel, Component, read_input_file
from aiomfac_py.s2as import smiles_to_components
from inorganic_component_basis import CATIONS, ANIONS
import generate_training_data as gtd
torch.set_num_threads(1)
CATION_NAMES = [n for n, _, _ in CATIONS]; ANION_NAMES = [n for n, _, _ in ANIONS]; ALL_IONS = CATION_NAMES + ANION_NAMES
IN_COLS = [f"m_{n}" for n in ALL_IONS] + ["T_K"]; OUT_COLS = ["ln_gamma_H2O"] + [f"ln_gamma_{n}" for n in ALL_IONS]
ION_SUB = {n: s for n, s, _ in CATIONS + ANIONS}; ION_Z = {n: z for n, _, z in CATIONS + ANIONS}
SUB2ION = {s: n for n, s in ION_SUB.items()}
fmt = lambda v: f"{np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}"
T0 = time.time()
print("ions:", ALL_IONS)

=== Finalized input vector spec (len=15) ===
  m_Na+      [mol/kg H2O]
  m_K+       [mol/kg H2O]
  m_NH4+     [mol/kg H2O]
  m_Ca2+     [mol/kg H2O]
  m_H+       [mol/kg H2O]
  m_Cl-      [mol/kg H2O]
  m_NO3-     [mol/kg H2O]
  m_SO4--    [mol/kg H2O]
  m_HSO4-    [mol/kg H2O]
  m_HCO3-    [mol/kg H2O]
  m_CO3--    [mol/kg H2O]
  m_IO3-     [mol/kg H2O]
  m_I-       [mol/kg H2O]
  m_OH-      [mol/kg H2O]
  T_K        [K]

=== Output vector spec (component ln(gamma), len=15) ===
  ln_gamma_H2O
  ln_gamma_Na+
  ln_gamma_K+
  ln_gamma_NH4+
  ln_gamma_Ca2+
  ln_gamma_H+
  ln_gamma_Cl-
  ln_gamma_NO3-
  ln_gamma_SO4--
  ln_gamma_HSO4-
  ln_gamma_HCO3-
  ln_gamma_CO3--
  ln_gamma_IO3-
  ln_gamma_I-
  ln_gamma_OH-

ions: ['Na+', 'K+', 'NH4+', 'Ca2+', 'H+', 'Cl-', 'NO3-', 'SO4--', 'HSO4-', 'HCO3-', 'CO3--', 'IO3-', 'I-', 'OH-']


## I.1 Data and splits

In [2]:
df = pd.read_parquet(HERE / "training_data.parquet")
df = df[df[OUT_COLS[1:]].notna().any(axis=1)].reset_index(drop=True)
HELD_OUT_COMBOS = [frozenset({"Ca2+", "CO3--"}), frozenset({"NH4+", "HSO4-"}), frozenset({"K+", "IO3-"})]
combo = df.apply(lambda r: frozenset(n for n in ALL_IONS if r[f"m_{n}"] > 0), axis=1)
is_gen = combo.isin(HELD_OUT_COMBOS).values
rest = np.where(~is_gen)[0]; p = np.random.RandomState(0).permutation(len(rest))
interp_idx = rest[p[:int(0.1 * len(rest))]]; train_idx = rest[p[int(0.1 * len(rest)):]]; gen_idx = np.where(is_gen)[0]
X_raw = np.concatenate([np.log1p(df[IN_COLS[:-1]].to_numpy(np.float32)), df[["T_K"]].to_numpy(np.float32)], 1)
Y_raw = df[OUT_COLS].to_numpy(np.float32); M = np.isfinite(Y_raw); Yz = np.nan_to_num(Y_raw)
print(f"rows {len(df)} | train {len(train_idx)}, interpolation test {len(interp_idx)}, generalization test {len(gen_idx)} "
      f"(held-out combos {[sorted(c) for c in HELD_OUT_COMBOS]})")
big = (np.abs(Yz) > 10) & M
print(f"labels with |ln γ| > 10: {big.sum()} of {M.sum()} ({100*big.sum()/M.sum():.2f} %); by split: "
      f"train {big[train_idx].sum()}, interp {big[interp_idx].sum()}, gen {big[gen_idx].sum()}")

rows 12000 | train 10420, interpolation test 1157, generalization test 423 (held-out combos [['CO3--', 'Ca2+'], ['HSO4-', 'NH4+'], ['IO3-', 'K+']])
labels with |ln γ| > 10: 73 of 52791 (0.14 %); by split: train 72, interp 1, gen 0


## I.2 Training: original protocol (replicated) vs. corrected protocol, 5 seeds each

In [3]:
class MLP(nn.Module):
    def __init__(self, n_in, n_out, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, n_out))
    def forward(self, x): return self.net(x)

class Surrogate(nn.Module):
    '''Wraps a trained MLP so that forward() returns ln γ in natural units (interface of the original scripts).'''
    def __init__(self, net, ym, ys):
        super().__init__(); self.net = net; self.register_buffer("ym", torch.tensor(ym)); self.register_buffer("ys", torch.tensor(ys))
    def forward(self, x): return self.net(x) * self.ys + self.ym

def masked_mse(p, t, m): return ((p - t) ** 2 * m).sum() / m.sum().clamp(min=1.0)

def train_original(seed):
    '''Replica of train_mlp_benchmark.train(): full batch, raw outputs, early stopping on the interpolation set.'''
    torch.manual_seed(seed)
    mu, sd = X_raw[train_idx].mean(0), X_raw[train_idx].std(0) + 1e-8
    Xt = torch.tensor((X_raw - mu) / sd); Yt = torch.tensor(Yz); Mt = torch.tensor(M.astype(np.float32))
    net = MLP(15, 15); opt = torch.optim.Adam(net.parameters(), lr=1e-3); best, st, bad = 1e9, None, 0
    tr, va = torch.tensor(train_idx), torch.tensor(interp_idx)
    for ep in range(400):
        net.train(); opt.zero_grad(); masked_mse(net(Xt[tr]), Yt[tr], Mt[tr]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = masked_mse(net(Xt[va]), Yt[va], Mt[va]).item()
        if v < best - 1e-6: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad > 25: break
    net.load_state_dict(st); net.eval()
    return Surrogate(net, np.zeros(15, np.float32), np.ones(15, np.float32)), (mu, sd)

def train_corrected(seed, max_epochs=500, patience=30, batch=256):
    rng = np.random.RandomState(seed); torch.manual_seed(seed)
    tr = train_idx.copy(); rng.shuffle(tr); nv = int(0.1 * len(tr)); val, fit = tr[:nv], tr[nv:]
    mu, sd = X_raw[fit].mean(0), X_raw[fit].std(0) + 1e-8
    ym = np.array([Y_raw[fit, j][M[fit, j]].mean() for j in range(15)], np.float32)
    ys = np.array([Y_raw[fit, j][M[fit, j]].std() + 1e-6 for j in range(15)], np.float32)
    Xt = torch.tensor((X_raw - mu) / sd); Yt = torch.tensor((Yz - ym) / ys * M); Mt = torch.tensor(M.astype(np.float32))
    net = MLP(15, 15); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); best, st, bad = 1e9, None, 0
    fit_t, val_t = torch.tensor(fit), torch.tensor(val)
    for ep in range(max_epochs):
        net.train(); perm = fit_t[torch.from_numpy(rng.permutation(len(fit)))]
        for s in range(0, len(perm), batch):
            i = perm[s:s + batch]; opt.zero_grad(); masked_mse(net(Xt[i]), Yt[i], Mt[i]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = masked_mse(net(Xt[val_t]), Yt[val_t], Mt[val_t]).item()
        if v < best - 1e-6: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(st); net.eval()
    return Surrogate(net, ym, ys), (mu, sd)

def predict(model, scaler, X):
    mu, sd = scaler
    with torch.no_grad(): return model(torch.tensor(((X - mu) / sd).astype(np.float32))).numpy()

def metrics(model, scaler, idx):
    P = predict(model, scaler, X_raw[idx]); E = np.abs(P - Yz[idx]); m = M[idx]
    per_sp = np.array([E[m[:, j], j].mean() if m[:, j].any() else np.nan for j in range(15)])
    small = m & (np.abs(Yz[idx]) <= 10)
    return {"pooled": E[m].mean(), "species_mean": np.nanmean(per_sp), "median": np.median(E[m]),
            "pooled_excl_big": E[small].mean(), "per_species": per_sp}

runs, models = [], {}
for proto, fn in [("original", train_original), ("corrected", train_corrected)]:
    for seed in range(5):
        mdl, sc = fn(seed); models[(proto, seed)] = (mdl, sc)
        for split, idx in [("train", train_idx), ("interp", interp_idx), ("gen", gen_idx)]:
            r = metrics(mdl, sc, idx); runs.append({"protocol": proto, "seed": seed, "split": split, **{k: v for k, v in r.items() if k != "per_species"}})
            if split != "train": runs[-1]["per_species"] = r["per_species"]
    print(f"  {proto} done ({time.time()-T0:.0f}s)")
R = pd.DataFrame(runs)
tab = R.groupby(["protocol", "split"])[["pooled", "species_mean", "median", "pooled_excl_big"]].agg(fmt)
display(tab)
ps = {(pr, sp): np.nanmean(np.stack(R[(R.protocol == pr) & (R.split == sp)].per_species.values), 0) for pr in ("original", "corrected") for sp in ("interp", "gen")}
display(pd.DataFrame({f"{pr} {sp}": v for (pr, sp), v in ps.items()}, index=OUT_COLS).round(3))
R.drop(columns="per_species").to_csv("P2_inorganic_runs.csv", index=False)

  original done (16s)


  corrected done (24s)


pooled   species_mean         median pooled_excl_big
protocol  split                                                              
corrected gen     0.058 ± 0.006  0.068 ± 0.007  0.026 ± 0.005   0.058 ± 0.006
          interp  0.069 ± 0.008  0.068 ± 0.006  0.025 ± 0.005   0.068 ± 0.008
          train   0.071 ± 0.010  0.070 ± 0.009  0.026 ± 0.005   0.065 ± 0.009
original  gen     0.096 ± 0.013  0.116 ± 0.017  0.048 ± 0.007   0.096 ± 0.013
          interp  0.087 ± 0.008  0.095 ± 0.008  0.043 ± 0.005   0.087 ± 0.008
          train   0.089 ± 0.011  0.096 ± 0.011  0.042 ± 0.006   0.084 ± 0.010

,original interp,original gen,corrected interp,corrected gen
ln_gamma_H2O,0.022,0.017,0.025,0.022
ln_gamma_Na+,0.047,NaN,0.024,NaN
ln_gamma_K+,0.066,0.054,0.034,0.023
ln_gamma_NH4+,0.081,0.057,0.048,0.034
ln_gamma_Ca2+,0.151,0.251,0.150,0.134
ln_gamma_H+,0.089,0.079,0.055,0.039
ln_gamma_Cl-,0.036,NaN,0.021,NaN
ln_gamma_NO3-,0.062,NaN,0.030,NaN
ln_gamma_SO4--,0.173,0.194,0.163,0.129
ln_gamma_HSO4-,0.084,0.053,0.050,0.026


## I.3 Comparison with real Fortran reference output (cases c005, c006)

In [4]:
from reference_utils import REF, parse_dump
fort = []
for c in ("c005", "c006"):
    case = read_input_file(REF / "carb_sulf" / "inputs" / f"input_{c}.txt")
    pts = parse_dump(REF / "carb_sulf" / "dumps" / f"debug_terms_{c}.txt.gz")
    sr = ActivityModel(case.components).mixture.sr
    for k, p in enumerate(pts):
        fr = np.asarray(case.fractions[k], float); w_water = 1 - fr[1:].sum() if len(fr) == len(case.components) else 1 - fr.sum()
        fr_s = fr[1:] if len(fr) == len(case.components) else fr
        tot = {n: 0.0 for n in ALL_IONS}
        for comp, f in zip(case.components[1:], fr_s):
            mw = sum((gtd.ION_MW[SUB2ION[s]] * q) for s, q in comp.subgroups)
            mol = f / w_water / mw
            for s, q in comp.subgroups: tot[SUB2ION[s]] += mol * q
        x = np.array([[*np.log1p([tot[n] for n in ALL_IONS]), float(case.T_K[k])]], np.float32)
        truth = {"ln_gamma_H2O": p["lnactcoeff_n"][0]}
        for j, s in enumerate(sr.cations): truth[f"ln_gamma_{SUB2ION[s]}"] = p["lnactcoeff_c"][j]
        for j, s in enumerate(sr.anions): truth[f"ln_gamma_{SUB2ION[s]}"] = p["lnactcoeff_a"][j]
        for seed in range(5):
            mdl, sc = models[("corrected", seed)]; pr = predict(mdl, sc, x)[0]
            for col, t in truth.items():
                if t > -9000: fort.append({"case": c, "point": k, "seed": seed, "species": col, "truth": t, "pred": pr[OUT_COLS.index(col)]})
fort = pd.DataFrame(fort); fort["abs_err"] = (fort.pred - fort.truth).abs()
print("Surrogate (corrected protocol) vs Fortran AIOMFAC reference, MAE in ln γ (mean ± sd over 5 seeds):")
display(fort.groupby(["case", "seed"]).abs_err.mean().groupby("case").agg(fmt))
display(fort.groupby("species").abs_err.mean().round(3).sort_values())
fort.to_csv("P2_fortran_comparison.csv", index=False)

Surrogate (corrected protocol) vs Fortran AIOMFAC reference, MAE in ln γ (mean ± sd over 5 seeds):


case
c005    0.144 ± 0.008
c006    0.103 ± 0.010
Name: abs_err, dtype: str

species
ln_gamma_H2O      0.027
ln_gamma_Na+      0.038
ln_gamma_HCO3-    0.058
ln_gamma_HSO4-    0.065
ln_gamma_OH-      0.068
ln_gamma_IO3-     0.077
ln_gamma_H+       0.120
ln_gamma_CO3--    0.165
ln_gamma_Ca2+     0.330
ln_gamma_SO4--    0.470
Name: abs_err, dtype: float64

## I.4 Bisulfate closed-form pipeline with the corrected surrogate

In [5]:
import io, contextlib
import closed_form_pipeline as cfp
samples = cfp.make_samples(300, seed=0)
with contextlib.redirect_stdout(io.StringIO()): orc = cfp.oracle_check(samples)
print(f"oracle check: max {np.max(orc):.2e}, mean {np.mean(orc):.2e}")
pipe = []
for seed in range(5):
    mdl, sc = models[("corrected", seed)]
    with contextlib.redirect_stdout(io.StringIO()): e3, e2 = cfp.surrogate_pipeline_check(mdl, sc, samples)
    scale = np.array([min(s["m_h_max"], s["m_sulf_max"]) for s in samples])
    pipe.append({"seed": seed, "mae_m_hso4": np.mean(e3), "rel_err_mean_%": 100 * np.mean(np.array(e3) / np.array([max(s["m_h_max"], s["m_sulf_max"]) for s in samples])),
                 "rel_err_median_%": 100 * np.median(np.array(e3) / np.array([max(s["m_h_max"], s["m_sulf_max"]) for s in samples]))})
pipe = pd.DataFrame(pipe); display(pipe.agg(fmt))

oracle check: max 8.88e-16, mean 3.67e-17

seed                2.000 ± 1.581
mae_m_hso4          0.009 ± 0.004
rel_err_mean_%      0.834 ± 0.374
rel_err_median_%    0.194 ± 0.046
dtype: str

## I.5 End-to-end cost: fixed system and varying ion combination (single CPU thread)

In [6]:
def med(fn, n, warm=5):
    for _ in range(warm): fn()
    t = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); t.append(time.perf_counter() - t0)
    return float(np.median(t))
mdl, sc = models[("corrected", 0)]; mu, sd = sc
rng = np.random.default_rng(1)
# (a) fixed system: NaCl, model built once, evaluate per point
comps, fr, tot, _ = gtd._build_system({("Na+", "Cl-"): 1.0}, 298.15); mA = ActivityModel(comps)
t_fixed = med(lambda: mA.evaluate([0.0] + list(fr), 298.15, basis="mass"), 300)
# (b) varying combination: build + evaluate per point (the Monte-Carlo / GSA use case)
combos = [gtd.sample_combo(rng) for _ in range(300)]
def aio(sm_T):
    c, f, t, _ = gtd._build_system(*sm_T); ActivityModel(c).evaluate([0.0] + list(f), sm_T[1], basis="mass")
t_vary = float(np.median([med(lambda s=s: aio(s), 1, warm=0) for s in combos]))
def vec(sm_T):
    _, _, tot, _ = gtd._build_system(*sm_T)
    return np.array([[*np.log1p([tot[n] for n in ALL_IONS]), sm_T[1]]], np.float32)
t_vec = float(np.median([med(lambda s=s: vec(s), 1, warm=0) for s in combos]))
x1 = vec(combos[0]); t_one = med(lambda: predict(mdl, sc, x1), 2000)
XB = np.concatenate([vec(s) for s in combos] * 34)[:10000]; t_batch = med(lambda: predict(mdl, sc, XB), 20) / len(XB)
print(f"AIOMFAC, fixed system (evaluate only):         {1e6*t_fixed:9.1f} µs/point")
print(f"AIOMFAC, varying combination (build+evaluate): {1e6*t_vary:9.1f} µs/point")
print(f"surrogate input-vector construction:            {1e6*t_vec:9.1f} µs/point")
print(f"surrogate forward, one at a time:               {1e6*t_one:9.1f} µs/point")
print(f"surrogate forward, batched (10 000):            {1e6*t_batch:9.2f} µs/point")
cost = {"fixed": t_fixed, "vary": t_vary, "vec": t_vec, "one": t_one, "batch": t_batch}
print(f"end-to-end speed-up, varying combination: one at a time {t_vary/(t_vec+t_one):.1f}x, batched {t_vary/(t_vec+t_batch):.0f}x; "
      f"fixed system: one at a time {t_fixed/(t_vec+t_one):.1f}x, batched {t_fixed/(t_vec+t_batch):.1f}x")
json.dump(cost, open("P2_cost.json", "w"), indent=1)

AIOMFAC, fixed system (evaluate only):             159.2 µs/point
AIOMFAC, varying combination (build+evaluate):    1925.2 µs/point
surrogate input-vector construction:                  9.8 µs/point
surrogate forward, one at a time:                    40.8 µs/point
surrogate forward, batched (10 000):                 0.45 µs/point
end-to-end speed-up, varying combination: one at a time 38.0x, batched 187x; fixed system: one at a time 3.1x, batched 15.5x


## I.6 Sobol analysis: surrogate vs. AIOMFAC directly (same samples)

In [7]:
from SALib.sample import sobol as sobol_sample
from SALib.analyze import sobol as sobol_analyze
CASES = [("NaCl", [("Na+", "Cl-")], ["ln_gamma_H2O", "ln_gamma_Na+"], 512),
         ("NaCl+NaNO3+KCl+KNO3", [("Na+", "Cl-"), ("Na+", "NO3-"), ("K+", "Cl-"), ("K+", "NO3-")], ["ln_gamma_H2O", "ln_gamma_Na+", "ln_gamma_K+"], 256),
         ("Ca(IO3)2+NaHSO4+NaHCO3", [("Ca2+", "IO3-"), ("Na+", "HSO4-"), ("Na+", "HCO3-")], ["ln_gamma_H2O", "ln_gamma_Ca2+", "ln_gamma_HSO4-"], 256)]
gsa = []
for name, salts, outs, N in CASES:
    prob = {"num_vars": len(salts) + 1, "names": [f"m_{c}{a}" for c, a in salts] + ["T_K"], "bounds": [[-3.0, 0.3]] * len(salts) + [[275.0, 325.0]]}
    Xs = sobol_sample.sample(prob, N, calc_second_order=False, seed=0)
    truth = np.full((len(Xs), 15), np.nan); xin = []
    for i, row in enumerate(Xs):
        sm = {s: 10 ** row[k] for k, s in enumerate(salts)}
        c, f, tot, _ = gtd._build_system(sm, row[-1]); r = gtd._evaluate(c, f, row[-1])
        if r is not None:
            truth[i] = [r[0].get(col, np.nan) for col in OUT_COLS]
        xin.append([*np.log1p([tot[n] for n in ALL_IONS]), row[-1]])
    xin = np.array(xin, np.float32)
    pred = np.mean([predict(*models[("corrected", s)], xin) for s in range(5)], 0)
    for col in outs:
        j = OUT_COLS.index(col); ok = np.isfinite(truth[:, j]).all()
        for src, Y in [("surrogate", pred[:, j]), ("AIOMFAC", truth[:, j])]:
            if src == "AIOMFAC" and not ok:
                Y = np.where(np.isfinite(Y), Y, np.nanmean(Y))      # rare failed points filled (reported below)
            Si = sobol_analyze.analyze(prob, Y.astype(float), calc_second_order=False, seed=0)
            for k, f in enumerate(prob["names"]):
                gsa.append({"case": name, "output": col, "source": src, "factor": f, "S1": Si["S1"][k], "S1_conf": Si["S1_conf"][k], "ST": Si["ST"][k]})
    print(f"  {name}: {np.isnan(truth[:, OUT_COLS.index(outs[0])]).sum()} failed AIOMFAC points of {len(Xs)} ({time.time()-T0:.0f}s)")
gsa = pd.DataFrame(gsa); gsa.to_csv("P2_sobol_surrogate_vs_aiomfac.csv", index=False)
piv = gsa.pivot_table(index=["case", "output", "factor"], columns="source", values=["S1", "ST"]).round(3)
display(piv)
# counter-ion claim: Na+ sensitivity to nitrate- vs chloride-paired salts
c2 = gsa[(gsa.case == "NaCl+NaNO3+KCl+KNO3") & (gsa.output == "ln_gamma_Na+")]
for src in ("surrogate", "AIOMFAC"):
    g = c2[c2.source == src].set_index("factor").S1
    print(f"{src:9s}: ΣS1(NO3- salts) = {g['m_Na+NO3-'] + g['m_K+NO3-']:.2f}, ΣS1(Cl- salts) = {g['m_Na+Cl-'] + g['m_K+Cl-']:.2f}")

  NaCl: 0 failed AIOMFAC points of 2048 (29s)


  NaCl+NaNO3+KCl+KNO3: 0 failed AIOMFAC points of 1792 (29s)


  Ca(IO3)2+NaHSO4+NaHCO3: 0 failed AIOMFAC points of 1536 (62s)


S1                ST  \
source                                           AIOMFAC surrogate AIOMFAC   
case                   output         factor                                 
Ca(IO3)2+NaHSO4+NaHCO3 ln_gamma_Ca2+  T_K          0.024    -0.021   0.069   
                                      m_Ca2+IO3-   0.173     0.330   0.732   
                                      m_Na+HCO3-  -0.000     0.036   0.467   
                                      m_Na+HSO4-   0.026     0.377   0.572   
                       ln_gamma_H2O   T_K          0.022    -0.003   0.106   
                                      m_Ca2+IO3-   0.240     0.488   0.362   
                                      m_Na+HCO3-   0.174     0.210   0.264   
                                      m_Na+HSO4-   0.446     0.137   0.589   
                       ln_gamma_HSO4- T_K          0.017    -0.001   0.042   
                                      m_Ca2+IO3-   0.183     0.776   0.411   
                                      m_Na+HCO3-  -0.003     0.009   0.164   
                                      m_Na+HSO4-   0.471     0.130   0.798   
NaCl                   ln_gamma_H2O   T_K          0.009     0.130   0.027   
                                      m_Na+Cl-     0.987     0.721   1.008   
                       ln_gamma_Na+   T_K          0.012     0.017   0.018   
                                      m_Na+Cl-     0.982     0.977   0.987   
NaCl+NaNO3+KCl+KNO3    ln_gamma_H2O   T_K          0.001     0.032   0.002   
                                      m_K+Cl-      0.046     0.124   0.059   
                                      m_K+NO3-     0.671     0.568   0.752   
                                      m_Na+Cl-     0.046     0.010   0.100   
                                      m_Na+NO3-    0.183     0.147   0.213   
                       ln_gamma_K+    T_K          0.010     0.009   0.013   
                                      m_K+Cl-      0.043     0.043   0.055   
                                      m_K+NO3-     0.392     0.427   0.449   
                                      m_Na+Cl-     0.015     0.005   0.059   
                                      m_Na+NO3-    0.414     0.389   0.513   
                       ln_gamma_Na+   T_K          0.042     0.041   0.048   
                                      m_K+Cl-      0.030     0.036   0.117   
                                      m_K+NO3-     0.312     0.345   0.431   
                                      m_Na+Cl-     0.004    -0.006   0.122   
                                      m_Na+NO3-    0.332     0.338   0.530   

                                                            
source                                           surrogate  
case                   output         factor                
Ca(IO3)2+NaHSO4+NaHCO3 ln_gamma_Ca2+  T_K            0.186  
                                      m_Ca2+IO3-     0.406  
                                      m_Na+HCO3-     0.158  
                                      m_Na+HSO4-     0.683  
                       ln_gamma_H2O   T_K            0.076  
                                      m_Ca2+IO3-     0.597  
                                      m_Na+HCO3-     0.301  
                                      m_Na+HSO4-     0.222  
                       ln_gamma_HSO4- T_K            0.011  
                                      m_Ca2+IO3-     0.831  
                                      m_Na+HCO3-     0.072  
                                      m_Na+HSO4-     0.177  
NaCl                   ln_gamma_H2O   T_K            0.266  
                                      m_Na+Cl-       0.843  
                       ln_gamma_Na+   T_K            0.022  
                                      m_Na+Cl-       0.983  
NaCl+NaNO3+KCl+KNO3    ln_gamma_H2O   T_K            0.050  
                                      m_K+Cl-        0.169  
                                      m_K+NO3-       0.655  
                                      m_Na+Cl-       0.051 

surrogate: ΣS1(NO3- salts) = 0.68, ΣS1(Cl- salts) = 0.03
AIOMFAC  : ΣS1(NO3- salts) = 0.64, ΣS1(Cl- salts) = 0.03


---
## II.1 Organic (R_emb release ensemble) and inorganic (corrected ensemble) surrogates

In [8]:
spec = importlib.util.spec_from_file_location("predict_remb", CKPT_DIR / "predict_remb.py"); pr_mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(pr_mod)
ORG = pr_mod.load_ensemble(); W_EMB = ORG[0][0].emb.weight.detach().numpy()
_cnt_cache = {}
def org_counts(smiles):
    if smiles not in _cnt_cache: _cnt_cache[smiles] = pr_mod.subgroup_counts(smiles)
    return _cnt_cache[smiles]
def organic_predict(smiles, w_bin, T_K):
    cnt, _ = org_counts(smiles); w = np.atleast_1d(w_bin).astype(float); T = np.broadcast_to(np.asarray(T_K, float), w.shape)
    out = []
    for net, em, es, ym, ys in ORG:
        x = np.concatenate([np.tile(cnt, (len(w), 1)), (np.column_stack([w, (T - 293.15) / 20]) - em) / es], 1).astype(np.float32)
        with torch.no_grad(): out.append(net(torch.tensor(x)).numpy() * ys + ym)
    return np.mean(out, 0)
def embedding(smiles): return org_counts(smiles)[0] @ W_EMB.T
INORG = [models[("corrected", s)] for s in range(5)]
def inorganic_predict(ion_molal, T_K):
    x = np.array([[*np.log1p([ion_molal.get(n, 0.0) for n in ALL_IONS]), T_K]], np.float32)
    return np.mean([predict(m, s, x)[0] for m, s in INORG], 0)

def salt_stoich(zc, za):
    g = math.gcd(abs(zc), abs(za)); return abs(za) // g, abs(zc) // g
def ternary_truth(smiles, cat, an, wtf_o, wtf_s, T_K):
    sub = pr_mod.smiles_to_components([smiles]).components[1].subgroups
    qc, qa = salt_stoich(ION_Z[cat], ION_Z[an])
    comps = [Component(1, "Water", ((16, 1),)), Component(2, "org", tuple(sub)), Component(3, "salt", ((ION_SUB[cat], qc), (ION_SUB[an], qa)))]
    mdl = ActivityModel(comps); res = mdl.evaluate([0.0, wtf_o, wtf_s], T_K, basis="mass")
    if not np.all(np.isfinite(res.ln_gamma)): return None
    sr = mdl.mixture.sr; nn_, nc = sr.n_neutral, sr.n_cation
    out = {"H2O": float(res.ln_gamma[0]), "organic": float(res.ln_gamma[1]),
           "cation": float(res.ln_gamma[nn_ + sr.cations.index(ION_SUB[cat])]),
           "anion": float(res.ln_gamma[nn_ + nc + sr.anions.index(ION_SUB[an])])}
    if min(out.values()) < -9000: return None
    w_w = 1 - wtf_o - wtf_s; n_salt = wtf_s / (qc * gtd.ION_MW[cat] + qa * gtd.ION_MW[an])
    return out, {"w_w": w_w, "m_cat": n_salt * qc / w_w, "m_an": n_salt * qa / w_w}
def option_a(smiles, cat, an, ctx, wtf_o, T_K):
    lgw_o, lgo = organic_predict(smiles, ctx["w_w"] / (ctx["w_w"] + wtf_o), T_K)[0]
    pi = inorganic_predict({cat: ctx["m_cat"], an: ctx["m_an"]}, T_K)
    return {"H2O": lgw_o + pi[0], "organic": lgo, "cation": pi[OUT_COLS.index(f"ln_gamma_{cat}")], "anion": pi[OUT_COLS.index(f"ln_gamma_{an}")]}
print("surrogates ready", f"({time.time()-T0:.0f}s)")

surrogates ready (62s)


## II.2 Option A error decomposition (324 ternary points, as in the original Table 5)

In [9]:
orgs = {d["name"]: d["smiles"] for d in json.load(open(HERE / "organics.json"))}
A_ORGS = [n for n in ["glucose", "citric acid", "glycerol", "sucrose", "levoglucosan", "ethanol"] if n in orgs]
A_SALTS = [("Na+", "Cl-"), ("NH4+", "SO4--"), ("Ca2+", "NO3-")]
def binary_truth_org(smiles, w_bin, T_K):
    m = ActivityModel(pr_mod.smiles_to_components([smiles]).components); return m.evaluate([w_bin, 1 - w_bin], T_K, basis="mass").ln_gamma[:2]
def binary_truth_salt(cat, an, m_cat, T_K):
    qc, _ = salt_stoich(ION_Z[cat], ION_Z[an]); c, f, tot, _ = gtd._build_system({(cat, an): m_cat / qc}, T_K); r = gtd._evaluate(c, f, T_K)
    return None if r is None else r[0]
rowsA = []
for o in A_ORGS:
    for cat, an in A_SALTS:
        for wo in (0.05, 0.15, 0.30):
            for ws in (0.02, 0.08):
                for T in (283.15, 298.15, 313.15):
                    g = ternary_truth(orgs[o], cat, an, wo, ws, T)
                    if g is None: continue
                    t, ctx = g; pa = option_a(orgs[o], cat, an, ctx, wo, T)
                    bo = binary_truth_org(orgs[o], ctx["w_w"] / (ctx["w_w"] + wo), T); bs = binary_truth_salt(cat, an, ctx["m_cat"], T)
                    if bs is None: continue
                    bsum = {"H2O": bo[0] + bs["ln_gamma_H2O"], "organic": bo[1], "cation": bs[f"ln_gamma_{cat}"], "anion": bs[f"ln_gamma_{an}"]}
                    for sp in ("H2O", "organic", "cation", "anion"):
                        rowsA.append({"organic": o, "salt": cat + an, "species": sp, "total": abs(t[sp] - pa[sp]),
                                      "own": abs(bsum[sp] - pa[sp]), "cross": abs(t[sp] - bsum[sp])})
A = pd.DataFrame(rowsA); A["sp2"] = A.species.replace({"cation": "ion", "anion": "ion"})
print(f"{A[A.species=='H2O'].shape[0]} ternary points")
display(A.groupby("sp2")[["total", "own", "cross"]].mean().round(3))
A.to_csv("P2_optionA_decomposition.csv", index=False)

324 ternary points


,total,own,cross
sp2,,,
H2O,0.031,0.024,0.007
ion,0.493,0.020,0.504
organic,0.362,0.041,0.358


## II.3 Option C residual model with separate calibration and test organics (3 seeds)

In [10]:
from rdkit import Chem
EXCLUDE = {"malonic acid", "ethanol"}
oxy = [d for d in json.load(open(HERE / "organics.json")) if d["name"] not in EXCLUDE
       and any(a.GetSymbol() == "O" for a in Chem.MolFromSmiles(d["smiles"]).GetAtoms())][:120]
TEST_O = [d["name"] for i, d in enumerate(oxy) if i % 6 == 0]; CAL_O = [d["name"] for i, d in enumerate(oxy) if i % 6 == 3]
TRAIN_O = [d["name"] for i, d in enumerate(oxy) if i % 6 not in (0, 3)]
SALTS_TRAIN = [("Na+", "Cl-"), ("Na+", "NO3-"), ("K+", "Cl-"), ("NH4+", "SO4--"), ("Ca2+", "NO3-"), ("H+", "Cl-"), ("Na+", "SO4--"), ("K+", "NO3-")]
SALTS_GEN = [("NH4+", "Cl-"), ("NH4+", "NO3-")]
print(f"organics: train {len(TRAIN_O)}, calibration {len(CAL_O)}, test {len(TEST_O)}; salts: train {len(SALTS_TRAIN)}, held out {len(SALTS_GEN)}")
def feat(smiles, cat, an, wo, ws, T, ctx):
    return np.concatenate([embedding(smiles), [wo, ws, (T - 293.15) / 20, np.log1p(ctx["m_cat"]), np.log1p(ctx["m_an"])],
                           [float(n == cat) for n in CATION_NAMES], [float(n == an) for n in ANION_NAMES]]).astype(np.float32)
def build(onames, salts):
    rows = []
    for o in onames:
        smi = orgs[o]
        for cat, an in salts:
            for wo in (0.08, 0.20, 0.30):
                for ws in (0.03, 0.08):
                    for T in (283.15, 313.15):
                        g = ternary_truth(smi, cat, an, wo, ws, T)
                        if g is None: continue
                        t, ctx = g; pa = option_a(smi, cat, an, ctx, wo, T)
                        rows.append({"organic": o, "salt": cat + an, "x": feat(smi, cat, an, wo, ws, T, ctx),
                                     **{f"t_{k}": v for k, v in t.items()}, **{f"a_{k}": v for k, v in pa.items()}})
    return pd.DataFrame(rows)
D_train = build(TRAIN_O, SALTS_TRAIN); D_us = build(TRAIN_O, SALTS_GEN)
D_uo = build(TEST_O, SALTS_TRAIN); D_both = build(TEST_O, SALTS_GEN); D_cal = build(CAL_O, SALTS_TRAIN + SALTS_GEN)
print({k: len(v) for k, v in dict(train=D_train, unseen_salt=D_us, unseen_org=D_uo, both=D_both, cal=D_cal).items()}, f"({time.time()-T0:.0f}s)")
SP = ["H2O", "organic", "cation", "anion"]

organics: train 80, calibration 20, test 20; salts: train 8, held out 2


{'train': 7680, 'unseen_salt': 1920, 'unseen_org': 1920, 'both': 480, 'cal': 2400} (102s)


In [11]:
class ResidualMLP(nn.Module):
    def __init__(self, n_in, h=96):
        super().__init__(); self.net = nn.Sequential(nn.Linear(n_in, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 4))
    def forward(self, x): return self.net(x)
p = np.random.RandomState(0).permutation(len(D_train)); n15 = int(0.15 * len(D_train))
I_TEST, I_VAL, I_FIT = p[:n15], p[n15:2 * n15], p[2 * n15:]
Xtr = np.stack(D_train.x.values); Ytr = np.stack([D_train[f"t_{s}"] - D_train[f"a_{s}"] for s in SP], 1).astype(np.float32)
def train_resid(seed):
    rng = np.random.RandomState(seed); torch.manual_seed(seed)
    xm, xs = Xtr[I_FIT].mean(0), Xtr[I_FIT].std(0) + 1e-6; ym, ys = Ytr[I_FIT].mean(0), Ytr[I_FIT].std(0) + 1e-6
    Xn = torch.tensor((Xtr - xm) / xs); Yn = torch.tensor((Ytr - ym) / ys)
    net = ResidualMLP(Xtr.shape[1]); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); L = nn.MSELoss()
    best, st, bad = 1e9, None, 0; fit_t, val_t = torch.tensor(I_FIT), torch.tensor(I_VAL)
    for ep in range(800):
        net.train(); perm = fit_t[torch.from_numpy(rng.permutation(len(I_FIT)))]
        for s in range(0, len(perm), 256):
            i = perm[s:s + 256]; opt.zero_grad(); L(net(Xn[i]), Yn[i]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = L(net(Xn[val_t]), Yn[val_t]).item()
        if v < best - 1e-6: best, st, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad > 50: break
    net.load_state_dict(st); net.eval()
    def f(X):
        with torch.no_grad(): return net(torch.tensor(((X - xm) / xs).astype(np.float32))).numpy() * ys + ym
    return f
RES = [train_resid(s) for s in range(3)]

# OOD guard in the R_emb embedding space: distance to nearest training organic
E_tr = np.stack([embedding(orgs[o]) for o in TRAIN_O]); e_m, e_s = E_tr.mean(0), E_tr.std(0) + 1e-6
def nn_dist(o): return float(np.min(np.linalg.norm((E_tr - e_m) / e_s - (embedding(orgs[o]) - e_m) / e_s, axis=1)))

def evaluate(Dset, f, guard_t=None):
    X = np.stack(Dset.x.values); r = f(X); out = {}
    for i, s in enumerate(SP):
        t = Dset[f"t_{s}"].values; a = Dset[f"a_{s}"].values; c = a + r[:, i]
        if s == "organic" and guard_t is not None:
            keep = np.array([nn_dist(o) <= guard_t for o in Dset.organic]); c = np.where(keep, c, a)
        out[s] = (np.abs(t - a).mean(), np.abs(t - c).mean())
    return out

thr_list, rows = [], []
for seed, f in enumerate(RES):
    # calibrate the guard on the CALIBRATION organics only
    r_cal = f(np.stack(D_cal.x.values))[:, 1]; per = []
    for o, g in D_cal.assign(r=r_cal).groupby("organic"):
        ea = np.abs(g.t_organic - g.a_organic).mean(); ec = np.abs(g.t_organic - (g.a_organic + g.r)).mean()
        per.append((nn_dist(o), ec < ea))
    d = np.array([x[0] for x in per]); h = np.array([x[1] for x in per]); best_t, best_sc = np.inf, -1
    for t in np.r_[np.unique(d), np.inf]:
        sc = h[d <= t].sum() + (~h[d > t]).sum()
        if sc > best_sc: best_sc, best_t = sc, t
    thr_list.append((best_t, best_sc, len(d)))
    for name, Dset in [("interpolation", D_train.iloc[I_TEST]), ("unseen salt", D_us), ("unseen organic", D_uo), ("unseen organic and salt", D_both)]:
        for guard in (False, True):
            if guard and name in ("interpolation", "unseen salt"): continue
            ev = evaluate(Dset, f, best_t if guard else None)
            rows.append({"seed": seed, "case": name + (" (guarded)" if guard else ""), **{f"{s} A": ev[s][0] for s in SP}, **{f"{s} C": ev[s][1] for s in SP}})
C_res = pd.DataFrame(rows)
print("guard thresholds (calibration organics):", [(round(t, 2), f"{sc}/{n}") for t, sc, n in thr_list])
summ = C_res.groupby("case", sort=False).mean(numeric_only=True).drop(columns="seed")
for s in SP: summ[f"{s} reduction %"] = 100 * (1 - summ[f"{s} C"] / summ[f"{s} A"])
display(summ.round(3))
test_d = {o: nn_dist(o) for o in TEST_O}
print("test organics beyond the median threshold:", [o for o, v in test_d.items() if v > np.median([t for t, _, _ in thr_list])])
C_res.to_csv("P2_optionC_runs.csv", index=False)

guard thresholds (calibration organics): [(np.float64(3.41), '20/20'), (np.float64(3.41), '20/20'), (np.float64(3.41), '20/20')]


,H2O A,organic A,cation A,anion A,H2O C,organic C,cation C,anion C,H2O reduction %,organic reduction %,cation reduction %,anion reduction %
case,,,,,,,,,,,,
interpolation,0.045,0.761,1.020,0.530,0.001,0.025,0.014,0.010,98.062,96.721,98.582,98.101
unseen salt,0.034,0.626,0.624,0.533,0.006,0.066,0.098,0.083,81.157,89.460,84.246,84.455
unseen organic,0.047,0.956,1.040,0.526,0.002,0.070,0.038,0.028,95.633,92.672,96.297,94.743
unseen organic (guarded),0.047,0.956,1.040,0.526,0.002,0.335,0.038,0.028,95.633,64.994,96.297,94.743
unseen organic and salt,0.036,0.744,0.629,0.532,0.007,0.094,0.103,0.095,80.792,87.387,83.675,82.228
unseen organic and salt (guarded),0.036,0.744,0.629,0.532,0.007,0.305,0.103,0.095,80.792,58.952,83.675,82.228


test organics beyond the median threshold: ['dianin’s compound', 'ergocalciferol']


## II.4 Worked examples: water + malonic acid + (NH4)2SO4 and water + ethanol + KCl (298.15 K)

In [12]:
THR = float(np.median([t for t, _, _ in thr_list]))
def worked(org, cat, an, grid):
    smi = orgs.get(org) or {"malonic acid": "OC(=O)CC(=O)O", "ethanol": "CCO"}[org]
    orgs[org] = smi; d = nn_dist(org); rows = []
    for wo, ws in grid:
        g = ternary_truth(smi, cat, an, wo, ws, 298.15)
        if g is None: continue
        t, ctx = g; pa = option_a(smi, cat, an, ctx, wo, 298.15); x = feat(smi, cat, an, wo, ws, 298.15, ctx)[None]
        r = np.mean([f(x)[0] for f in RES], 0)
        pc = {s: pa[s] + r[i] for i, s in enumerate(SP)}
        if d > THR: pc["organic"] = pa["organic"]
        rows.append({"wo": wo, "ws": ws, **{f"t_{s}": t[s] for s in SP}, **{f"a_{s}": pa[s] for s in SP}, **{f"c_{s}": pc[s] for s in SP}})
    return pd.DataFrame(rows), d
mal, d_mal = worked("malonic acid", "NH4+", "SO4--", [(a, b) for a in (0.03, 0.07, 0.11, 0.15) for b in (0.02, 0.05, 0.08)])
eth, d_eth = worked("ethanol", "K+", "Cl-", [(a, 0.03) for a in np.linspace(0.05, 0.45, 13)])
print(f"guard threshold (median over seeds) {THR:.2f}; nn distance: malonic acid {d_mal:.2f}, ethanol {d_eth:.2f}")
for name, Dw in [("malonic acid + (NH4)2SO4", mal), ("ethanol + KCl", eth)]:
    print(f"\n{name} (n = {len(Dw)}):  MAE  Option A  ->  Option C (guarded)")
    for s in SP: print(f"  ln γ {s:8s} {np.abs(Dw[f't_{s}'] - Dw[f'a_{s}']).mean():.3f}  ->  {np.abs(Dw[f't_{s}'] - Dw[f'c_{s}']).mean():.3f}")
gpm = lambda D, p: np.exp(0.5 * (D[f"{p}_cation"] + D[f"{p}_anion"]))
for p_ in ("a", "c"):
    e = np.abs(gpm(eth, p_) - gpm(eth, "t")); print(f"ethanol + KCl γ± MAE, {'Option A' if p_ == 'a' else 'Option C'}: {e.mean():.3f} ({100*(e / gpm(eth, 't')).mean():.1f} %)")
mal.to_csv("P2_worked_malonic.csv", index=False); eth.to_csv("P2_worked_ethanol_kcl.csv", index=False)
print(f"total runtime {time.time()-T0:.0f}s")

guard threshold (median over seeds) 3.41; nn distance: malonic acid 2.26, ethanol 1.07

malonic acid + (NH4)2SO4 (n = 12):  MAE  Option A  ->  Option C (guarded)
  ln γ H2O      0.015  ->  0.005
  ln γ organic  0.103  ->  0.117
  ln γ cation   0.064  ->  0.049
  ln γ anion    0.150  ->  0.047

ethanol + KCl (n = 13):  MAE  Option A  ->  Option C (guarded)
  ln γ H2O      0.025  ->  0.003
  ln γ organic  0.116  ->  0.029
  ln γ cation   0.973  ->  0.039
  ln γ anion    0.693  ->  0.031
ethanol + KCl γ± MAE, Option A: 0.956 (52.6 %)
ethanol + KCl γ± MAE, Option C: 0.081 (3.6 %)
total runtime 120s


## III. Per-organic effects, saved models and figures

In [13]:
# per-organic effect of the organic-species correction (calibration + test organics, seed-averaged)
per_org = []
for nm, Dset in [("calibration", D_cal), ("test", pd.concat([D_uo, D_both]))]:
    r = np.mean([f(np.stack(Dset.x.values))[:, 1] for f in RES], 0)
    for o, g in Dset.assign(r=r).groupby("organic"):
        ea = np.abs(g.t_organic - g.a_organic).mean(); ec = np.abs(g.t_organic - (g.a_organic + g.r)).mean()
        per_org.append({"set": nm, "organic": o, "nn_dist": nn_dist(o), "mae_A": ea, "mae_C": ec, "reduction_%": 100 * (1 - ec / ea)})
per_org = pd.DataFrame(per_org); per_org.to_csv("P2_per_organic_correction.csv", index=False)
print("correction helps for", (per_org.mae_C < per_org.mae_A).sum(), "of", len(per_org), "unseen organics")
display(per_org.sort_values("nn_dist", ascending=False).head(8).round(3))
ps_df = pd.DataFrame({f"{pr} {sp}": v for (pr, sp), v in ps.items()}, index=OUT_COLS); ps_df.to_csv("P2_per_species_mae.csv")
os.makedirs("p2_models", exist_ok=True)
for s in range(5):
    mdl, (mu_, sd_) = models[("corrected", s)]
    torch.save({"state_dict": mdl.state_dict(), "x_mean": mu_, "x_std": sd_, "in_cols": IN_COLS, "out_cols": OUT_COLS,
                "input_transform": "log1p(molality) for ions, raw T_K"}, f"p2_models/inorganic_seed{s}.pt")
print("saved p2_models/inorganic_seed*.pt")

correction helps for 40 of 40 unseen organics


,set,organic,nn_dist,mae_A,mae_C,reduction_%
30,test,dianin’s compound,4.672,3.078,0.194,93.697
32,test,ergocalciferol,4.220,2.554,0.336,86.827
12,calibration,beta-estradiol-3-benzoate,3.406,2.019,0.162,91.954
13,calibration,cholecalciterol,3.008,2.431,0.222,90.863
17,calibration,phenolphthalein,2.554,1.598,0.139,91.302
18,calibration,salicin,2.541,1.004,0.056,94.472
25,test,3-pentadecylcyclohexanone,2.323,1.739,0.210,87.908
38,test,propylencarbonate,2.289,0.203,0.105,48.207


saved p2_models/inorganic_seed*.pt


In [14]:
import matplotlib; import matplotlib.pyplot as plt
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]; INK, MUTED, GRID = "#222222", "#6b6b6b", "#e6e6e6"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False, "savefig.dpi": 300, "savefig.bbox": "tight"})
panel = lambda ax, s: ax.text(-0.1, 1.04, s, transform=ax.transAxes, fontweight="bold", fontsize=10, color=INK)
lab = lambda c: c.replace("ln_gamma_", "").replace("--", "²⁻").replace("2+", "²⁺").replace("+", "⁺").replace("-", "⁻").replace("H2O", "H₂O").replace("NH4", "NH₄").replace("NO3", "NO₃").replace("SO4", "SO₄").replace("HSO4", "HSO₄").replace("HCO3", "HCO₃").replace("CO3", "CO₃").replace("IO3", "IO₃")
os.makedirs("figs", exist_ok=True)
# Fig 1: per-species interpolation MAE, original vs corrected protocol
fig, ax = plt.subplots(figsize=(6.6, 2.8)); x = np.arange(15); w = 0.38
ax.bar(x - w / 2, ps_df["original interp"], w, color=C[1], label="original protocol")
ax.bar(x + w / 2, ps_df["corrected interp"], w, color=C[0], label="corrected protocol")
ax.set_xticks(x, [lab(c) for c in OUT_COLS], rotation=45, ha="right", fontsize=8); ax.set_ylabel("Interpolation MAE, ln γ"); ax.legend(fontsize=8)
plt.savefig("figs/p2_fig1_species_mae.png"); plt.close()
# Fig 2: Sobol S1, surrogate vs AIOMFAC for two cases
fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.8))
for ax, (case, out, t) in zip(axes, [("NaCl+NaNO3+KCl+KNO3", "ln_gamma_Na+", "(a)"), ("Ca(IO3)2+NaHSO4+NaHCO3", "ln_gamma_HSO4-", "(b)")]):
    g = gsa[(gsa.case == case) & (gsa.output == out)].pivot(index="factor", columns="source", values="S1")
    y = np.arange(len(g)); ax.barh(y + 0.2, g["AIOMFAC"], 0.38, color=C[2], label="AIOMFAC"); ax.barh(y - 0.2, g["surrogate"], 0.38, color=C[0], label="surrogate")
    SALT = {"m_Na+NO3-": "NaNO₃", "m_Na+Cl-": "NaCl", "m_K+NO3-": "KNO₃", "m_K+Cl-": "KCl", "m_Na+HSO4-": "NaHSO₄", "m_Na+HCO3-": "NaHCO₃", "m_Ca2+IO3-": "Ca(IO₃)₂", "T_K": "T"}
    ax.set_yticks(y, [SALT.get(f, f) for f in g.index], fontsize=8); ax.set_xlabel(f"S₁, ln γ({lab(out)})"); panel(ax, t)
axes[0].legend(fontsize=8, loc="lower right"); plt.tight_layout(); plt.savefig("figs/p2_fig2_sobol.png"); plt.close()
# Fig 3: Option A vs C by generalization case, and per-organic correction effect vs distance
fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.9), gridspec_kw={"width_ratios": [1.3, 1]})
cases = ["interpolation", "unseen salt", "unseen organic", "unseen organic and salt"]; sp_lbl = {"H2O": "H₂O", "organic": "organic", "cation": "cation", "anion": "anion"}
ax = axes[0]; x = np.arange(4); w = 0.2
for k, s in enumerate(SP):
    ax.bar(x + (k - 1.5) * w, [summ.loc[c, f"{s} C"] / summ.loc[c, f"{s} A"] for c in cases], w, color=C[k], label=sp_lbl[s])
ax.set_xticks(x, ["interp.", "unseen\nsalt", "unseen\norganic", "unseen\nboth"], fontsize=8); ax.set_ylabel("MAE ratio, Option C / Option A")
ax.legend(fontsize=7, ncol=2); panel(ax, "(a)")
ax = axes[1]
for nm, mk, c in [("calibration", "o", C[6]), ("test", "s", C[1])]:
    g = per_org[per_org.set == nm]; ax.scatter(g.nn_dist, g["reduction_%"], marker=mk, s=22, color=c, label=f"{nm} organics")
ax.axhline(0, color=MUTED, lw=0.8); ax.set_xlabel("Distance to nearest training organic (R_emb space)"); ax.set_ylabel("Organic error reduction (%)")
ax.legend(fontsize=7); panel(ax, "(b)")
plt.tight_layout(); plt.savefig("figs/p2_fig3_option_c.png"); plt.close()
# Fig 4: worked examples
fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.9))
ax = axes[0]
for p_, c, ls, nm in [("t", INK, "-", "AIOMFAC"), ("a", C[1], "--", "Option A"), ("c", C[0], ":", "Option C")]:
    ax.plot(eth.wo, gpm(eth, p_), color=c, ls=ls, lw=2, marker="o" if p_ == "t" else None, ms=3, label=nm)
ax.set_xlabel("Ethanol mass fraction (KCl mass fraction 0.03)"); ax.set_ylabel("γ± (KCl)"); ax.legend(fontsize=8); panel(ax, "(a)")
ax = axes[1]
for k, s in enumerate(SP):
    ax.scatter(mal[f"t_{s}"], mal[f"a_{s}"], marker="x", s=18, color=C[k]); ax.scatter(mal[f"t_{s}"], mal[f"c_{s}"], marker="o", s=14, color=C[k], label=sp_lbl[s].replace("cation", "NH₄⁺").replace("anion", "SO₄²⁻"))
lim = [min(mal[[f"t_{s}" for s in SP]].min()) - 0.2, max(mal[[f"t_{s}" for s in SP]].max()) + 0.2]
ax.plot(lim, lim, color=MUTED, lw=0.8); ax.set_xlabel("AIOMFAC ln γ"); ax.set_ylabel("Predicted ln γ (×: A, ○: C)"); ax.legend(fontsize=7); panel(ax, "(b)")
plt.tight_layout(); plt.savefig("figs/p2_fig4_worked_examples.png"); plt.close()
print("figures written", f"({time.time()-T0:.0f}s)")

figures written (121s)


## IV. Comparison with the EMF-derived γ± of Lopes et al. (1999), water + ethanol + KCl, 298.15 K

Lopes et al. report Pitzer parameters (Table III, 25 °C; unaffected by the 2001 erratum, which corrects
only the Pitzer–Simonson parameters) and the Debye–Hückel coefficient A_φ (Table II) for 5, 10, 15 and 20 %
(w/w, salt-free) ethanol. These give the experimental γ± on the molal scale with the **mixed-solvent reference
state** (infinite dilution in the respective salt-free solvent). Model values (AIOMFAC, Option A, Option C) are
converted to the same reference state by dividing by each model's own γ± at infinite dilution
(m = 10⁻⁴ mol kg⁻¹) in the same solvent, as in Zuend et al. (2011, Fig. 7). Note that these data were part
of AIOMFAC's own parameter fit (Zuend et al., 2011, Table 2), so AIOMFAC itself is not independently
validated here; the test is whether the surrogate pipeline reproduces AIOMFAC's agreement with experiment.

In [15]:
LOPES = {0.05: (0.4112, 0.0393, 0.224, 0.0041), 0.10: (0.4335, 0.0527, 0.283, -0.0006),
         0.15: (0.4584, 0.0760, 0.116, -0.0104), 0.20: (0.4863, 0.0350, 0.181, 0.0105)}   # A_phi, beta0, beta1, C_phi at 25 C
def pitzer_ln_gpm(m, Aphi, b0, b1, cphi, alpha=2.0, b=1.2):
    sm = np.sqrt(m)
    f = -Aphi * (sm / (1 + b * sm) + (2 / b) * np.log(1 + b * sm))
    Bg = 2 * b0 + 2 * b1 / (alpha ** 2 * m) * (1 - (1 + alpha * sm - alpha ** 2 * m / 2) * np.exp(-alpha * sm))
    return f + m * Bg + m ** 2 * 1.5 * cphi
M_KCL = gtd.ION_MW["K+"] + gtd.ION_MW["Cl-"]
def comp_from_molal(xe, m):
    tot = 1.0 + m * M_KCL; return xe / tot, m * M_KCL / tot          # ethanol and salt mass fractions (1 kg solvent basis)
def ln_gpm_models(xe, m):
    wo, ws = comp_from_molal(xe, m); g = ternary_truth("CCO", "K+", "Cl-", wo, ws, 298.15)
    t, ctx = g; pa = option_a("CCO", "K+", "Cl-", ctx, wo, 298.15); x = feat("CCO", "K+", "Cl-", wo, ws, 298.15, ctx)[None]
    r = np.mean([f(x)[0] for f in RES], 0); pc = {s: pa[s] + r[i] for i, s in enumerate(SP)}
    h = lambda d: 0.5 * (d["cation"] + d["anion"])
    return h(t), h(pa), h(pc)
MOLAL = [0.1, 0.2, 0.5, 1.0, 1.5, 2.0]
lop = []
for xe, par in LOPES.items():
    ref = ln_gpm_models(xe, 1e-4)
    for m in MOLAL:
        mod = ln_gpm_models(xe, m)
        lop.append({"ethanol_wt": xe, "m": m, "exp": np.exp(pitzer_ln_gpm(m, *par)),
                    "AIOMFAC": np.exp(mod[0] - ref[0]), "Option A": np.exp(mod[1] - ref[1]), "Option C": np.exp(mod[2] - ref[2]),
                    "lnref_AIOMFAC": ref[0], "lnref_A": ref[1], "lnref_C": ref[2],
                    "ln_AIOMFAC": mod[0], "ln_A": mod[1], "ln_C": mod[2]})
lop = pd.DataFrame(lop); lop.to_csv("P2_lopes_comparison.csv", index=False)
for k in ("AIOMFAC", "Option A", "Option C"):
    e = (lop[k] - lop.exp).abs(); print(f"{k:9s} vs Lopes et al. (1999): γ± MAE {e.mean():.3f} ({100*(e/lop.exp).mean():.1f} %), max {100*(e/lop.exp).max():.1f} %")
for k in ("Option A", "Option C"):
    e = (lop[k] - lop.AIOMFAC).abs(); print(f"{k:9s} vs AIOMFAC (same points):    γ± MAE {e.mean():.3f} ({100*(e/lop.AIOMFAC).mean():.1f} %)")
# where does the deviation of Option C from AIOMFAC come from? ln(γ±,C/γ±,AIOMFAC) in the mixed-solvent reference
# state = (error at finite m, own reference state) − (error at m = 10⁻⁴)
for k, c in (("Option A", "A"), ("Option C", "C")):
    fin = (lop[f"ln_{c}"] - lop.ln_AIOMFAC).abs().mean(); r0 = (lop[f"lnref_{c}"] - lop.lnref_AIOMFAC).groupby(lop.ethanol_wt).first().abs()
    print(f"{k}: |Δln γ±| at finite m (unconverted) {fin:.3f}; at m = 1e-4 (reference term) {r0.mean():.3f} "
          f"(by ethanol fraction: {', '.join(f'{v:.3f}' for v in r0)})")
display(lop[["ethanol_wt", "m", "exp", "AIOMFAC", "Option A", "Option C"]].round(3))
fig, axes = plt.subplots(1, 4, figsize=(8.6, 2.5), sharey=True)
for ax, (xe, g) in zip(axes, lop.groupby("ethanol_wt")):
    ax.plot(g.m, g.exp, "o", color=INK, ms=4, label="Lopes et al. (1999)")
    ax.plot(g.m, g.AIOMFAC, "-", color=C[2], lw=2, label="AIOMFAC")
    ax.plot(g.m, g["Option C"], ":", color=C[0], lw=2, label="Option C")
    ax.plot(g.m, g["Option A"], "--", color=C[1], lw=1.5, label="Option A")
    ax.set_title(f"{int(round(100*xe))} % ethanol", fontsize=9); ax.set_xlabel("m(KCl) / mol kg⁻¹")
axes[0].set_ylabel("γ± (mixed-solvent reference)"); axes[0].legend(fontsize=6.5)
plt.tight_layout(); plt.savefig("figs/p2_fig5_lopes.png"); plt.close()

AIOMFAC   vs Lopes et al. (1999): γ± MAE 0.011 (1.9 %), max 5.9 %
Option A  vs Lopes et al. (1999): γ± MAE 0.081 (14.6 %), max 33.6 %
Option C  vs Lopes et al. (1999): γ± MAE 0.050 (8.7 %), max 17.1 %
Option A  vs AIOMFAC (same points):    γ± MAE 0.093 (16.7 %)
Option C  vs AIOMFAC (same points):    γ± MAE 0.062 (10.8 %)
Option A: |Δln γ±| at finite m (unconverted) 0.383; at m = 1e-4 (reference term) 0.536 (by ethanol fraction: 0.239, 0.436, 0.634, 0.833)
Option C: |Δln γ±| at finite m (unconverted) 0.022; at m = 1e-4 (reference term) 0.121 (by ethanol fraction: 0.076, 0.111, 0.132, 0.165)


,ethanol_wt,m,exp,AIOMFAC,Option A,Option C
0,0.05,0.1,0.756,0.757,0.792,0.793
1,0.05,0.2,0.701,0.700,0.746,0.749
2,0.05,0.5,0.629,0.626,0.682,0.685
3,0.05,1.0,0.579,0.578,0.642,0.635
4,0.05,1.5,0.558,0.558,0.628,0.600
5,0.05,2.0,0.548,0.550,0.624,0.582
6,0.10,0.1,0.750,0.737,0.789,0.784
7,0.10,0.2,0.697,0.676,0.742,0.734
8,0.10,0.5,0.629,0.598,0.678,0.666
9,0.10,1.0,0.583,0.549,0.640,0.608
